<a href="https://colab.research.google.com/github/matthew-ngzc/AI-Safety-Module/blob/main/Week_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repository

In [1]:
try:
    ! git clone https://github.com/NayMyatMin/CS427_SMU
    HOME_DIR = "./CS427_SMU/week7/"
except:
    print('Already clone!!!')

fatal: destination path 'CS427_SMU' already exists and is not an empty directory.


Exercise 0: Computing correlation between 2 features, using a randomly created dataset to calcualte correlation using spearman coefficient

In [2]:
import numpy as np
import scipy.stats

sex = np.array([0,0,0,0,0,1,1,1,1,1]) #0 for M; 1 for F
ethnicity = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 0]) #0 for native; 1 for non-native
highestdegree = np.array([1,2,1,1,2,2,1,0,2,1]) #0 for none; 1 for high-school; 2 for university
jobtype = np.array([0,0,0,1,1,2,2,1,2,0]) #0 for board; 1 for healthcare; 2 for education)

print("sex and ethnicity: {}".format(scipy.stats.spearmanr(sex, ethnicity)))
print("sex and highestdegree: {}".format(scipy.stats.spearmanr(sex, highestdegree)))
print("sex and jobtype: {}".format(scipy.stats.spearmanr(sex, jobtype)))


sex and ethnicity: SignificanceResult(statistic=np.float64(0.0), pvalue=np.float64(1.0))
sex and highestdegree: SignificanceResult(statistic=np.float64(-0.11547005383792514), pvalue=np.float64(0.7507502763206226))
sex and jobtype: SignificanceResult(statistic=np.float64(0.590168890850654), pvalue=np.float64(0.07248247888882482))


# Exercise 5
For a Neural Network that predicts whether an individual makes more than 50K annually,
1. Apply suppressing to train 2 new models.
- 1 suppressing the gender attribute
- 1 suppressing gender and the most correlated attribute to gender (using spearman coefficient)

2. Compare the accuracy of the 3 models
3. Compare the fairness score of the 3 models using $|P(>50K|M) - P(>50K|F)|$

In [3]:
def split_test_by_gender(original_test_data, suppressed_test_data, gender_idx):
    """
    original_test_data   : full dataset (with gender column)
    suppressed_test_data : dataset used by model (gender removed)
    gender_idx           : index of gender in original dataset

    Returns:
        female_samples, male_samples
        (using suppressed features, but split by original gender)
    """

    female_samples = []
    male_samples = []

    for i in range(len(original_test_data)):
        x_original = original_test_data[i]
        x_suppressed = suppressed_test_data[i]

        # split using original gender
        if x_original[gender_idx] < 0.5:
            female_samples.append(x_suppressed) # append the suppressed one to maintain dimensions
        else:
            male_samples.append(x_suppressed)

    return female_samples, male_samples

In [4]:
# store the results for displaying later
results = []

In [5]:
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import numpy as np
import ast
import scipy.stats



class CensusNet(nn.Module):
    def __init__(self, num_of_features):
        super().__init__()
        self.fc1 = nn.Linear(num_of_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 8)
        self.fc5 = nn.Linear(8, 4)
        self.fc6 = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        x = F.relu(x)
        x = self.fc5(x)
        x = F.relu(x)
        x = self.fc6(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def save_model(model, name):
    torch.save(model.state_dict(), name)


def load_model(model_class, name, num_of_features):
    model = model_class(num_of_features)
    model.load_state_dict(torch.load(name))

    return model


def train(model, dataloader, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()

    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # if batch % 100 == 0:
        #     loss, current = loss.item(), batch * len(x)
        #     print('loss: {:.4f} [{}/{}]'.format(loss, current, size))


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    accuracy = correct / size
    print('Test Error: \n Accuracy: {:.2f}%, Avg loss: {:.4f}\n'.format(100 * accuracy, loss))
    return accuracy


def get_loader(input_file, label_file, kwargs, input_data=None):
    if input_data is None:
        input_file = open(input_file, 'r')
        input_data = []
        for line in input_file.readlines():
            input_data.append(ast.literal_eval(line))
        input_data = np.array(input_data)

    label_file = open(label_file, 'r')
    label_data = np.array(ast.literal_eval(label_file.readline()))

    input_data_tensor = torch.Tensor(input_data)
    label_data_tensor = torch.Tensor(label_data).type(torch.LongTensor)
    dataset = TensorDataset(input_data_tensor, label_data_tensor)
    data_loader = torch.utils.data.DataLoader(dataset, **kwargs)

    return input_data, label_data, data_loader


def train_model(num_of_features, train_loader, test_loader, device, name):
    model = CensusNet(num_of_features).to(device)

    optimizer = optim.SGD(model.parameters(), lr=0.1)
    num_of_epochs = 20

    for epoch in range(num_of_epochs):
        print('\n------------- Epoch {} -------------\n'.format(epoch))
        train(model, train_loader, nn.CrossEntropyLoss(), optimizer, device)
        test(model, test_loader, nn.CrossEntropyLoss(), device)

    save_model(model, name)


# returns the index of the feature which has the highest spearman correlation with gender
def find_corr(num_of_features, gender_idx, train_input_data):
    max_rho, max_i = 0.0, -1

    for i in range(num_of_features):
        if i == gender_idx: continue

        feature_i = train_input_data[:,i]
        feature_j = train_input_data[:,gender_idx]

        rho, pvalue = scipy.stats.spearmanr(feature_i, feature_j)
        print(' i = {}, corr = {}'.format(i, rho))

        if abs(rho) > max_rho:
            max_rho = abs(rho)
            max_i = i

    return max_i


def generate_x(size, lower, upper):
    x = np.random.rand(size)
    x = (upper - lower) * x + lower

    return x


device = 'cpu'
train_kwargs = {'batch_size': 100}
test_kwargs = {'batch_size': 1000}

train_input_data, train_label_data, train_loader = get_loader(HOME_DIR + 'exercise4/train/input.txt', HOME_DIR + 'exercise4/train/label.txt', train_kwargs)
test_input_data, test_label_data, test_loader = get_loader(HOME_DIR + 'exercise4/test/input.txt', HOME_DIR + 'exercise4/test/label.txt', test_kwargs)

num_of_features = 13
gender_idx = 8

train_model(num_of_features, train_loader, test_loader, device, HOME_DIR + 'exercise4/censusOriginal.pt')
model = load_model(CensusNet, HOME_DIR + 'exercise4/censusOriginal.pt', num_of_features)
test(model, test_loader, nn.CrossEntropyLoss(), device)

lower = np.zeros(num_of_features)
upper = np.ones(num_of_features)

num_of_samples = 10000
female_samples, male_samples = split_test_by_gender(test_input_data, test_input_data, gender_idx)


print('no of samples female = {}, male = {}'.format(len(female_samples), len(male_samples)))

def test_fairness(model, samples):
    cnt = 0
    for x in samples:
        x = x.copy()
        x = x.reshape(1, -1)
        x_tensor = torch.Tensor(x)
        if model(x_tensor).argmax(1).item() == 0: # >50K
            cnt += 1
    return cnt


model = load_model(CensusNet, HOME_DIR + 'exercise4/censusOriginal.pt', num_of_features)

acc_og = test(model, test_loader, nn.CrossEntropyLoss(), device)

female_cnt = test_fairness(model, female_samples)
male_cnt = test_fairness(model, male_samples)

pr_female = female_cnt / len(female_samples)
pr_male = male_cnt / len(male_samples)

fairness_og = abs(pr_male - pr_female) * 100

print('|Pr(>50K|Male) - Pr(>50K|Female)| = {}'.format(fairness_og))
results.append({
    "Model": "Original",
    "Accuracy(%)": acc_og * 100,
    "|Pr(>50K|M) - Pr(>50K|F)|": fairness_og
})



------------- Epoch 0 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5472


------------- Epoch 1 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5409


------------- Epoch 2 -------------

Test Error: 
 Accuracy: 82.08%, Avg loss: 0.3830


------------- Epoch 3 -------------

Test Error: 
 Accuracy: 81.94%, Avg loss: 0.3708


------------- Epoch 4 -------------

Test Error: 
 Accuracy: 82.19%, Avg loss: 0.3637


------------- Epoch 5 -------------

Test Error: 
 Accuracy: 82.59%, Avg loss: 0.3584


------------- Epoch 6 -------------

Test Error: 
 Accuracy: 83.07%, Avg loss: 0.3535


------------- Epoch 7 -------------

Test Error: 
 Accuracy: 82.95%, Avg loss: 0.3532


------------- Epoch 8 -------------

Test Error: 
 Accuracy: 83.21%, Avg loss: 0.3491


------------- Epoch 9 -------------

Test Error: 
 Accuracy: 83.25%, Avg loss: 0.3473


------------- Epoch 10 -------------

Test Error: 
 Accuracy: 83.59%, Avg loss: 0.3421


------------- Epoch 11 -------

## 1. Supress Gender

In [6]:
# remove gender col from train and test datasets
train_input_no_gender = np.delete(train_input_data, gender_idx, axis=1)
test_input_no_gender  = np.delete(test_input_data, gender_idx, axis=1)

num_features_no_gender = num_of_features - 1

# make new dataloaders. Instead of using input file we use input data cos we alr have that in "train_input_no_gender"
_, _, train_loader_no_gender = get_loader(
    None,
    HOME_DIR + 'exercise4/train/label.txt',
    train_kwargs,
    input_data=train_input_no_gender
)

_, _, test_loader_no_gender = get_loader(
    None,
    HOME_DIR + 'exercise4/test/label.txt',
    test_kwargs,
    input_data=test_input_no_gender
)

# train and test model
train_model(num_features_no_gender, train_loader_no_gender, test_loader_no_gender, device, HOME_DIR + 'exercise4/censusNoGender.pt')
model_no_gender = load_model(CensusNet, HOME_DIR + 'exercise4/censusNoGender.pt', num_features_no_gender)
test(model_no_gender, test_loader_no_gender, nn.CrossEntropyLoss(), device)

# Evaluate fairness
female_samples, male_samples = split_test_by_gender(test_input_data, test_input_no_gender, gender_idx)
female_count = test_fairness(model_no_gender, female_samples)
male_count   = test_fairness(model_no_gender, male_samples)

pr_female = female_count / len(female_samples)
pr_male   = male_count / len(male_samples)

print('|Pr(>50K|Male) - Pr(>50K|Female)| = {}'.format(abs(pr_male - pr_female) * 100))

# Store results
acc_no_gender = test(model_no_gender, test_loader_no_gender, nn.CrossEntropyLoss(), device)
fairness_no_gender = abs(pr_male - pr_female) * 100
results.append({
    "Model": "No Gender",
    "Accuracy(%)": acc_no_gender * 100,
    "|Pr(>50K|M) - Pr(>50K|F)|": fairness_no_gender
})



------------- Epoch 0 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5477


------------- Epoch 1 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5477


------------- Epoch 2 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5476


------------- Epoch 3 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5476


------------- Epoch 4 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5475


------------- Epoch 5 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5474


------------- Epoch 6 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5470


------------- Epoch 7 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5450


------------- Epoch 8 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.4680


------------- Epoch 9 -------------

Test Error: 
 Accuracy: 80.62%, Avg loss: 0.3888


------------- Epoch 10 -------------

Test Error: 
 Accuracy: 82.49%, Avg loss: 0.3651


------------- Epoch 11 -------

## 2. Supress Gender + most correlated feature

In [9]:
# Find most correlated feature
most_corr = find_corr(num_of_features, gender_idx, train_input_data)
print('most correlated feature = {}'.format(most_corr))
feature_names = [
    "age",
    "workclass",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country"
]
most_corr_name = feature_names[most_corr]
print("Most correlated feature:", most_corr_name)

# remove col
if most_corr > gender_idx:
    most_corr -= 1

train_input_suppress_2 = np.delete(train_input_no_gender, most_corr, axis=1)
test_input_suppress_2 = np.delete(test_input_no_gender, most_corr, axis=1)

num_features_suppress_2 = num_features_no_gender - 1

# make new dataloaders
_, _, train_loader_suppress_2 = get_loader(
    None,
    HOME_DIR + 'exercise4/train/label.txt',
    train_kwargs,
    input_data=train_input_suppress_2
)

_, _, test_loader_suppress_2 = get_loader(
    None,
    HOME_DIR + 'exercise4/test/label.txt',
    test_kwargs,
    input_data=test_input_suppress_2
)

# train and test model
filename = 'exercise4/censusSuppress2.pt'
train_model(num_features_suppress_2, train_loader_suppress_2, test_loader_suppress_2, device, HOME_DIR + filename)
model_suppress_2 = load_model(CensusNet, HOME_DIR + filename, num_features_suppress_2)
test(model_suppress_2, test_loader_suppress_2, nn.CrossEntropyLoss(), device)

# Evaluate fairness
female_samples, male_samples = split_test_by_gender(test_input_data, test_input_suppress_2, gender_idx)
female_count = test_fairness(model_suppress_2, female_samples)
male_count   = test_fairness(model_suppress_2, male_samples)

pr_female = female_count / len(female_samples)
pr_male   = male_count / len(male_samples)

print('|Pr(>50K|Male) - Pr(>50K|Female)| = {}'.format(abs(pr_male - pr_female) * 100))

# Store results
acc_suppress_2 = test(model_suppress_2, test_loader_suppress_2, nn.CrossEntropyLoss(), device)
fairness_suppress_2 = abs(pr_male - pr_female) * 100
results.append({
    "Model": f"No Gender + No {most_corr_name}",
    "Accuracy(%)": acc_suppress_2 * 100,
    "|Pr(>50K|M) - Pr(>50K|F)|": fairness_suppress_2
})

 i = 0, corr = 0.10037341705509416
 i = 1, corr = 0.07087071753014831
 i = 2, corr = 0.02757212668964831
 i = 3, corr = 0.006283152076072141
 i = 4, corr = -0.399709262159814
 i = 5, corr = -0.030397397179514264
 i = 6, corr = -0.16125903956877638
 i = 7, corr = -0.10751775564628759
 i = 9, corr = 0.06664600258767539
 i = 10, corr = 0.04215426465744419
 i = 11, corr = 0.2649405933542167
 i = 12, corr = -0.005190720882382154
most correlated feature = 4
Most correlated feature: marital-status

------------- Epoch 0 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5472


------------- Epoch 1 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5450


------------- Epoch 2 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.4708


------------- Epoch 3 -------------

Test Error: 
 Accuracy: 80.92%, Avg loss: 0.4016


------------- Epoch 4 -------------

Test Error: 
 Accuracy: 80.57%, Avg loss: 0.4002


------------- Epoch 5 -------------

Test Error: 
 Accuracy: 81

I wanted to compare with deleting every other col alongside gender to see the pattern, completely generated by AI

In [11]:
# ------------------------------------------
# Feature names (must align with dataset)
# ------------------------------------------

feature_names = [
    "age",
    "workclass",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country"
]

# ------------------------------------------
# Remove gender first
# ------------------------------------------

train_input_no_gender = np.delete(train_input_data, gender_idx, axis=1)
test_input_no_gender  = np.delete(test_input_data, gender_idx, axis=1)

num_features_no_gender = num_of_features - 1

feature_names_no_gender = feature_names.copy()
feature_names_no_gender.pop(gender_idx)

# ------------------------------------------
# Loop through every remaining feature
# ------------------------------------------

for i, feature_name in enumerate(feature_names_no_gender):

    print(f"\nTraining: No Gender + No {feature_name}")

    # Remove this feature
    train_input_tmp = np.delete(train_input_no_gender, i, axis=1)
    test_input_tmp  = np.delete(test_input_no_gender, i, axis=1)

    num_features_tmp = train_input_tmp.shape[1]

    # Build dataloaders
    _, _, train_loader_tmp = get_loader(
        None,
        HOME_DIR + 'exercise4/train/label.txt',
        train_kwargs,
        input_data=train_input_tmp
    )

    _, _, test_loader_tmp = get_loader(
        None,
        HOME_DIR + 'exercise4/test/label.txt',
        test_kwargs,
        input_data=test_input_tmp
    )

    # Train model
    filename = f"exercise4/census_NoGender_No_{feature_name}.pt"

    train_model(
        num_features_tmp,
        train_loader_tmp,
        test_loader_tmp,
        device,
        HOME_DIR + filename
    )

    # Load model
    model_tmp = load_model(
        CensusNet,
        HOME_DIR + filename,
        num_features_tmp
    )

    # ------------------------------------------
    # Accuracy
    # ------------------------------------------

    acc_tmp = test(
        model_tmp,
        test_loader_tmp,
        nn.CrossEntropyLoss(),
        device
    )

    # ------------------------------------------
    # Fairness
    # ------------------------------------------

    female_samples, male_samples = split_test_by_gender(
        test_input_data,      # original for gender
        test_input_tmp,       # suppressed features
        gender_idx
    )

    female_count = test_fairness(model_tmp, female_samples)
    male_count   = test_fairness(model_tmp, male_samples)

    pr_female = female_count / len(female_samples)
    pr_male   = male_count / len(male_samples)

    dp_gap = abs(pr_male - pr_female) * 100

    print('|Pr(>50K|Male) - Pr(>50K|Female)| = {}'.format(dp_gap))

    # ------------------------------------------
    # Store results
    # ------------------------------------------

    results.append({
        "Model": f"No Gender + No {feature_name}",
        "Accuracy(%)": acc_tmp * 100,
        "|Pr(>50K|M) - Pr(>50K|F)|": dp_gap
    })


Training: No Gender + No age

------------- Epoch 0 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5477


------------- Epoch 1 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5476


------------- Epoch 2 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5475


------------- Epoch 3 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5473


------------- Epoch 4 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5467


------------- Epoch 5 -------------

Test Error: 
 Accuracy: 76.38%, Avg loss: 0.5327


------------- Epoch 6 -------------

Test Error: 
 Accuracy: 81.24%, Avg loss: 0.4005


------------- Epoch 7 -------------

Test Error: 
 Accuracy: 82.38%, Avg loss: 0.3775


------------- Epoch 8 -------------

Test Error: 
 Accuracy: 82.82%, Avg loss: 0.3687


------------- Epoch 9 -------------

Test Error: 
 Accuracy: 83.23%, Avg loss: 0.3640


------------- Epoch 10 -------------

Test Error: 
 Accuracy: 83.39%, Avg loss: 0.3589




Display Results

In [12]:
import pandas as pd

df = pd.DataFrame(results)
print(df)

                            Model  Accuracy(%)  |Pr(>50K|M) - Pr(>50K|F)|
0                        Original    84.577114                  16.661321
1                       No Gender    83.704932                  21.105013
2   No Gender + No marital-status    84.233155                  20.725366
3              No Gender + No age    84.398993                  18.716196
4        No Gender + No workclass    84.650820                  18.732930
5        No Gender + No education    84.681531                  18.300883
6    No Gender + No education-num    84.466556                  18.117853
7   No Gender + No marital-status    83.907622                  19.322708
8       No Gender + No occupation    84.245439                  18.833700
9     No Gender + No relationship    76.377372                   0.000000
10            No Gender + No race    84.337571                  18.943066
11    No Gender + No capital-gain    83.145998                  18.788576
12    No Gender + No capital-loss    8

From the results we can see that gender was not the main cause of the bias, removing it along with its most correlated feature did not improve the fairness.

For runs 9 and 13, it seems like the model is converging at always predicting <=50K, so there is 0 difference in the probabilities. The accuracy still remains relatively high due to the class split in the test set

Looking at the other features, it seems like the "relationship" feature should be closely related to gender, since its values are:
- Wife
- Own-child
- Husband
- Not-in-family
- Other-relative
- Unmarried

Which have gender specific categories. Its likely that it was not chosen as the most correlated feature by spearman coefficient because the categorical feature was encoded as ordinal integers, but spearman only detects monotonic rank relationships. Because the ordering is arbitrary, it's relationship is not captured well